In [2]:
!nvidia-smi
!python -V

Sat Apr 18 03:52:41 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   40C    P8             12W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [3]:
from pathlib import Path
import os
import sys

REPO_URL = "https://github.com/likhithashree01-beep/Habitat.git"
REPO_BRANCH = "main"
PROJECT_ROOT = Path("/content/Habitat")

os.environ["HABITAT_REPO_URL"] = REPO_URL
os.environ["HABITAT_REPO_BRANCH"] = REPO_BRANCH
os.environ["HABITAT_PROJECT_ROOT"] = str(PROJECT_ROOT)

print("REPO_URL:", REPO_URL)
print("REPO_BRANCH:", REPO_BRANCH)
print("PROJECT_ROOT:", PROJECT_ROOT)

REPO_URL: https://github.com/likhithashree01-beep/Habitat.git
REPO_BRANCH: main
PROJECT_ROOT: /content/Habitat


In [4]:
%%bash
set -e

REPO_URL="$HABITAT_REPO_URL"
REPO_BRANCH="$HABITAT_REPO_BRANCH"
PROJECT_ROOT="$HABITAT_PROJECT_ROOT"

if [[ -z "$REPO_URL" ]]; then
  echo "Set REPO_URL before running this cell."
  exit 1
fi

if [ ! -d "$PROJECT_ROOT/.git" ]; then
  rm -rf "$PROJECT_ROOT"
  git clone --branch "$REPO_BRANCH" "$REPO_URL" "$PROJECT_ROOT"
else
  cd "$PROJECT_ROOT"
  git fetch origin "$REPO_BRANCH"
  git checkout "$REPO_BRANCH"
  git pull --ff-only origin "$REPO_BRANCH"
fi

cd "$PROJECT_ROOT"
git rev-parse --abbrev-ref HEAD
git rev-parse HEAD

Your branch is up to date with 'origin/main'.
Already up to date.
main
7250b457550d7f47777627e3aab97f6a9eb2646a


From https://github.com/likhithashree01-beep/Habitat
 * branch            main       -> FETCH_HEAD
Already on 'main'
From https://github.com/likhithashree01-beep/Habitat
 * branch            main       -> FETCH_HEAD


In [5]:
expected = [
    "code",
    "configs",
    "datasets",
    "notebooks",
    "outputs",
]

missing = [name for name in expected if not (PROJECT_ROOT / name).exists()]
if missing:
    print("Missing repo folders:", missing)
    print("Creating them so environment setup can continue.")

for rel in [
    "code/agents",
    "code/runners",
    "code/utils",
    "configs",
    "datasets",
    "outputs/logs",
    "outputs/videos",
    "notebooks",
]:
    (PROJECT_ROOT / rel).mkdir(parents=True, exist_ok=True)

os.chdir(PROJECT_ROOT)

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

print("PROJECT_ROOT:", PROJECT_ROOT)
print("cwd:", Path.cwd())
print("Top-level folders:", sorted(p.name for p in PROJECT_ROOT.iterdir()))

PROJECT_ROOT: /content/Habitat
cwd: /content/Habitat
Top-level folders: ['.git', '.gitignore', 'HabitatAI (1).ipynb', 'Miniconda3-latest-Linux-x86_64.sh', 'code', 'configs', 'datasets', 'notebooks', 'outputs']


In [6]:
%%bash
set -e

if [ ! -d "/opt/conda" ]; then
  wget -q https://repo.anaconda.com/miniconda/Miniconda3-latest-Linux-x86_64.sh
  bash Miniconda3-latest-Linux-x86_64.sh -b -p /opt/conda
fi

source /opt/conda/etc/profile.d/conda.sh
conda tos accept --override-channels --channel https://repo.anaconda.com/pkgs/main
conda tos accept --override-channels --channel https://repo.anaconda.com/pkgs/r
conda --version

accepted Terms of Service for https://repo.anaconda.com/pkgs/main
accepted Terms of Service for https://repo.anaconda.com/pkgs/r
conda 26.1.1


In [7]:
import os
os.environ["PATH"] = "/opt/conda/bin:" + os.environ["PATH"]
print(os.environ["PATH"].split(":")[:4])

['/opt/conda/bin', '/opt/bin', '/usr/local/cuda/bin', '/usr/local/sbin']


In [8]:
%%bash
set -e
source /opt/conda/etc/profile.d/conda.sh

if ! conda env list | awk '{print $1}' | grep -qx habitatEnv; then
  conda create -y -n habitatEnv -c conda-forge python=3.9
fi

conda activate habitatEnv

pip install -U pip
pip install "pillow==10.4.0"
conda install -y -c conda-forge -c aihabitat habitat-sim withbullet headless
pip install "git+https://github.com/facebookresearch/habitat-lab.git#subdirectory=habitat-lab"


python -c "import habitat_sim; print('habitat_sim OK', habitat_sim.__version__)"
python -c "import habitat; print('habitat OK', habitat.__version__)"
python -c "import PIL; print('Pillow', PIL.__version__)"

Jupyter detected...
2 channel Terms of Service accepted
Channels:
 - conda-forge
 - aihabitat
 - defaults
Platform: linux-64
Solving environment: done

# All requested packages already installed.

  Cloning https://github.com/facebookresearch/habitat-lab.git to /tmp/pip-req-build-n7cqp4gj
  Resolved https://github.com/facebookresearch/habitat-lab.git to commit a9c8df586d649972e55500a0fbaae1952b1c3483
  Installing build dependencies: started
  Installing build dependencies: finished with status 'done'
  Getting requirements to build wheel: started
  Getting requirements to build wheel: finished with status 'done'
  Preparing metadata (pyproject.toml): started
  Preparing metadata (pyproject.toml): finished with status 'done'
  Created wheel for habitat-lab: filename=habitat_lab-0.3.3-py3-none-any.whl size=467316 sha256=3e6cfdf45dc649e689e8b89bf9fd565d1e7f5849ff43fffaae579a28f69ea32c
  Stored in directory: /tmp/pip-ephem-wheel-cache-lmjkq9i0/wheels/79/fc/52/971fac582740089ffb4a50d623c1b6



==> WARNING: A newer version of conda exists. <==
    current version: 26.1.1
    latest version: 26.3.2

Please update conda by running

    $ conda update -n base -c defaults conda


  Running command git clone --filter=blob:none --quiet https://github.com/facebookresearch/habitat-lab.git /tmp/pip-req-build-n7cqp4gj
Gym has been unmaintained since 2022 and does not support NumPy 2.0 amongst other critical functionality.
Please upgrade to Gymnasium, the maintained drop-in replacement of Gym, or contact the authors of your software and request that they upgrade.
See the migration guide at https://gymnasium.farama.org/introduction/migration_guide/ for additional information.
pybullet build time: Jan 29 2025 23:20:52
PluginManager::Manager: duplicate static plugin StbImageImporter, ignoring
PluginManager::Manager: duplicate static plugin GltfImporter, ignoring
PluginManager::Manager: duplicate static plugin BasisImporter, ignoring
PluginManager::Manager: duplicate static plugin AssimpI

## Configure Headless EGL

Colab needs an explicit EGL vendor file so Habitat-Sim can render without a display.


In [9]:
%%bash
set -e

NVIDIA_EGL_LIB=$(find /usr -name "libEGL_nvidia.so.0" 2>/dev/null | head -n 1)
echo "Using: $NVIDIA_EGL_LIB"

cat > /tmp/nvidia_egl.json <<EOF
{
  "file_format_version": "1.0.0",
  "ICD": { "library_path": "$NVIDIA_EGL_LIB" }
}
EOF

echo "EGL ICD created at /tmp/nvidia_egl.json"

Using: /usr/lib64-nvidia/libEGL_nvidia.so.0
EGL ICD created at /tmp/nvidia_egl.json


In [10]:
import glob
import os

lib = glob.glob("/usr/**/libEGL_nvidia.so.0", recursive=True)[0]
os.environ["__EGL_VENDOR_LIBRARY_FILENAMES"] = "/tmp/nvidia_egl.json"
os.environ["LD_LIBRARY_PATH"] = os.path.dirname(lib) + ":" + os.environ.get("LD_LIBRARY_PATH", "")
os.environ["NVIDIA_DRIVER_CAPABILITIES"] = "all"

print("EGL forced to:", lib)
print("EGL JSON:", os.environ["__EGL_VENDOR_LIBRARY_FILENAMES"])

EGL forced to: /usr/lib64-nvidia/libEGL_nvidia.so.0
EGL JSON: /tmp/nvidia_egl.json


In [11]:
%%bash
set -e
source /opt/conda/etc/profile.d/conda.sh
conda activate habitatEnv

python - <<'PY'
import habitat
import habitat_sim

print('Habitat-Lab import success')
print('Habitat-Sim import success')
print('habitat version:', habitat.__version__)
print('habitat_sim version:', habitat_sim.__version__)
PY

Habitat-Lab import success
Habitat-Sim import success
habitat version: 0.3.3
habitat_sim version: 0.3.3


Gym has been unmaintained since 2022 and does not support NumPy 2.0 amongst other critical functionality.
Please upgrade to Gymnasium, the maintained drop-in replacement of Gym, or contact the authors of your software and request that they upgrade.
See the migration guide at https://gymnasium.farama.org/introduction/migration_guide/ for additional information.
pybullet build time: Jan 29 2025 23:20:52
PluginManager::Manager: duplicate static plugin StbImageImporter, ignoring
PluginManager::Manager: duplicate static plugin GltfImporter, ignoring
PluginManager::Manager: duplicate static plugin BasisImporter, ignoring
PluginManager::Manager: duplicate static plugin AssimpImporter, ignoring
PluginManager::Manager: duplicate static plugin AnySceneImporter, ignoring
PluginManager::Manager: duplicate static plugin AnyImageImporter, ignoring


In [12]:
%%bash
set -e
source /opt/conda/etc/profile.d/conda.sh
conda activate habitatEnv

SCENE_DIR=/content/Habitat/datasets/test_scenes
mkdir -p "$SCENE_DIR"
cd "$SCENE_DIR"

if [ ! -f "skokloster-castle.glb" ] || [ ! -s "skokloster-castle.glb" ]; then
  echo "Downloading habitat-test-scenes.zip..."
  rm -f skokloster-castle.glb skokloster-castle.navmesh habitat-test-scenes.zip
  wget -q http://dl.fbaipublicfiles.com/habitat/habitat-test-scenes.zip
  unzip -q -o habitat-test-scenes.zip
  find . -name "skokloster-castle.glb" -exec mv {} . \;
  find . -name "skokloster-castle.navmesh" -exec mv {} . \;
  rm -f habitat-test-scenes.zip
fi

ls -lh skokloster-castle.glb skokloster-castle.navmesh

-rw-r--r-- 1 root root 37M Jun 10  2021 skokloster-castle.glb
-rw-r--r-- 1 root root 28K Feb 18  2019 skokloster-castle.navmesh


In [13]:
%%bash
set -e
source /opt/conda/etc/profile.d/conda.sh
conda activate habitatEnv

python - <<'PY'
import os
import habitat_sim
from habitat_sim import Configuration, SimulatorConfiguration
from habitat_sim.agent import AgentConfiguration
from habitat_sim.sensor import CameraSensorSpec, SensorType

scene_path = "/content/Habitat/datasets/test_scenes/skokloster-castle.glb"
assert os.path.exists(scene_path), f"Missing scene: {scene_path}"

sim_cfg = SimulatorConfiguration()
sim_cfg.scene_id = scene_path
sim_cfg.enable_physics = False

rgb_spec = CameraSensorSpec()
rgb_spec.uuid = "rgb"
rgb_spec.sensor_type = SensorType.COLOR
rgb_spec.resolution = [256, 256]
rgb_spec.position = [0.0, 1.5, 0.0]
rgb_spec.hfov = 90.0

agent_cfg = AgentConfiguration()
agent_cfg.sensor_specifications = [rgb_spec]

cfg = Configuration(sim_cfg, [agent_cfg])
sim = habitat_sim.Simulator(cfg)
obs = sim.get_sensor_observations()
rgb = obs["rgb"]

print("Obs keys:", list(obs.keys()))
print("RGB shape:", rgb.shape)
print("RGB dtype:", rgb.dtype)
print("RGB min/max:", rgb.min(), rgb.max())

sim.close()
print("Direct-config headless render OK")
PY

Renderer: Tesla T4/PCIe/SSE2 by NVIDIA Corporation
OpenGL version: 4.6.0 NVIDIA 580.82.07
Using optional features:
    GL_ARB_vertex_array_object
    GL_ARB_separate_shader_objects
    GL_ARB_robustness
    GL_ARB_texture_storage
    GL_ARB_texture_view
    GL_ARB_framebuffer_no_attachments
    GL_ARB_invalidate_subdata
    GL_ARB_texture_storage_multisample
    GL_ARB_multi_bind
    GL_ARB_direct_state_access
    GL_ARB_get_texture_sub_image
    GL_ARB_texture_filter_anisotropic
    GL_KHR_debug
    GL_KHR_parallel_shader_compile
    GL_NV_depth_buffer_float
Using driver workarounds:
    no-forward-compatible-core-context
    nv-egl-incorrect-gl11-function-pointers
    no-layout-qualifiers-on-old-glsl
    nv-zero-context-profile-mask
    nv-implementation-color-read-format-dsa-broken
    nv-cubemap-inconsistent-compressed-image-size
    nv-cubemap-broken-full-compressed-image-query
    nv-compressed-block-size-in-bits
Obs keys: ['rgb']
RGB shape: (256, 256, 4)
RGB dtype: uint8
RGB min

[03:53:42:105257]:[Warning]:[Metadata] SceneDatasetAttributes.cpp(107)::addNewSceneInstanceToDataset : Dataset : 'default' : Lighting Layout Attributes 'no_lights' specified in Scene Attributes but does not exist in dataset, so creating default.
[03:53:42:105539]:[Warning]:[Scene] SemanticScene.h(331)::checkFileExists : ::loadSemanticSceneDescriptor: File `/content/Habitat/datasets/test_scenes/skokloster-castle.scn` does not exist.  Aborting load.
[03:53:42:105568]:[Warning]:[Scene] SemanticScene.cpp(123)::loadSemanticSceneDescriptor : SSD File Naming Issue! Neither SemanticAttributes-provided name : `/content/Habitat/datasets/test_scenes/skokloster-castle.scn` nor constructed filename : `/content/Habitat/datasets/test_scenes/info_semantic.json` exist on disk.
[03:53:42:105585]:[Error]:[Scene] SemanticScene.cpp(139)::loadSemanticSceneDescriptor : SSD Load Failure! File with SemanticAttributes-provided name `/content/Habitat/datasets/test_scenes/skokloster-castle.scn` exists but failed 

In [14]:
%%bash
set -e
source /opt/conda/etc/profile.d/conda.sh
conda activate habitatEnv

python - <<'PY'
from habitat.config.default import get_config

cfg = get_config("benchmark/nav/pointnav/pointnav_hm3d.yaml")

print("=== DATASET CONFIG ===")
print("type:", cfg.habitat.dataset.type)
print("split:", cfg.habitat.dataset.split)
print("scenes_dir:", cfg.habitat.dataset.scenes_dir)
print("data_path:", cfg.habitat.dataset.data_path)

print("\n=== SENSOR SETUP ===")
print(list(cfg.habitat.simulator.agents.main_agent.sim_sensors.keys()))

print("\nPointNav config load OK")
PY

=== DATASET CONFIG ===
type: PointNav-v1
split: train
scenes_dir: data/scene_datasets
data_path: data/datasets/pointnav/hm3d/v1/{split}/{split}.json.gz

=== SENSOR SETUP ===
['rgb_sensor', 'depth_sensor']

PointNav config load OK


Gym has been unmaintained since 2022 and does not support NumPy 2.0 amongst other critical functionality.
Please upgrade to Gymnasium, the maintained drop-in replacement of Gym, or contact the authors of your software and request that they upgrade.
See the migration guide at https://gymnasium.farama.org/introduction/migration_guide/ for additional information.
pybullet build time: Jan 29 2025 23:20:52
PluginManager::Manager: duplicate static plugin StbImageImporter, ignoring
PluginManager::Manager: duplicate static plugin GltfImporter, ignoring
PluginManager::Manager: duplicate static plugin BasisImporter, ignoring
PluginManager::Manager: duplicate static plugin AssimpImporter, ignoring
PluginManager::Manager: duplicate static plugin AnySceneImporter, ignoring
PluginManager::Manager: duplicate static plugin AnyImageImporter, ignoring


In [15]:
from pathlib import Path

dataset_dirs = [
    PROJECT_ROOT / "datasets" / "scene_datasets",
    PROJECT_ROOT / "datasets" / "pointnav",
    PROJECT_ROOT / "datasets" / "hm3d",
]

for path in dataset_dirs:
    path.mkdir(parents=True, exist_ok=True)
    print(path)

print("\nDataset directories are ready. Download cells can target these paths in the next notebook.")

/content/Habitat/datasets/scene_datasets
/content/Habitat/datasets/pointnav
/content/Habitat/datasets/hm3d

Dataset directories are ready. Download cells can target these paths in the next notebook.


Install Habitat Lab and Setup

In [16]:
%%bash
source /opt/conda/etc/profile.d/conda.sh && conda activate habitatEnv

# Clone habitat-lab v0.3.3 if not already present
if [ ! -d /content/habitat-lab-v033 ]; then
    cd /content
    git clone --depth 1 --branch v0.3.3 https://github.com/facebookresearch/habitat-lab.git habitat-lab-v033
    pip uninstall habitat-lab -y
    cd habitat-lab-v033
    pip install -e habitat-lab/
fi

# Install pybullet (needed for humanoid_utils imports)
pip install pybullet 2>&1 | tail -1

python -c "import habitat; print('habitat-lab:', habitat.__version__)"
python -c "import habitat_sim; print('habitat-sim:', habitat_sim.__version__)"

habitat-lab: 0.3.3
habitat-sim: 0.3.3


Gym has been unmaintained since 2022 and does not support NumPy 2.0 amongst other critical functionality.
Please upgrade to Gymnasium, the maintained drop-in replacement of Gym, or contact the authors of your software and request that they upgrade.
See the migration guide at https://gymnasium.farama.org/introduction/migration_guide/ for additional information.
pybullet build time: Jan 29 2025 23:20:52
PluginManager::Manager: duplicate static plugin StbImageImporter, ignoring
PluginManager::Manager: duplicate static plugin GltfImporter, ignoring
PluginManager::Manager: duplicate static plugin BasisImporter, ignoring
PluginManager::Manager: duplicate static plugin AssimpImporter, ignoring
PluginManager::Manager: duplicate static plugin AnySceneImporter, ignoring
PluginManager::Manager: duplicate static plugin AnyImageImporter, ignoring


Mount Drive

In [20]:
from google.colab import drive
drive.mount('/content/drive')

from pathlib import Path
DRIVE_DATA = Path('/content/drive/MyDrive/HabitatData')
print("Data folder:", DRIVE_DATA)

Mounted at /content/drive
Data folder: /content/drive/MyDrive/HabitatData


In [ ]:
# from google.colab import drive
# drive.flush_and_unmount()
